In [1]:
!pip uninstall -y google-generativeai google-ai-generativelanguage
!pip install -q --upgrade google-generativeai neo4j

Found existing installation: google-generativeai 0.8.5
Uninstalling google-generativeai-0.8.5:
  Successfully uninstalled google-generativeai-0.8.5
Found existing installation: google-ai-generativelanguage 0.6.15
Uninstalling google-ai-generativelanguage-0.6.15:
  Successfully uninstalled google-ai-generativelanguage-0.6.15
DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621
    pytz (>dev)
         ~^
  DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621
  DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near

In [1]:
import google.generativeai as genai
import neo4j
import json
import os

/var/folders/xp/9xhg9lln1n527rx10rl74pwr0000gn/T/ipykernel_1970/1214644645.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
print("Bağlantı bilgilerini tanımlayın:\n")

# Neo4j Aura bilgileri (AYNI KALIR)
NEO4J_URI = "neo4j://localhost:7687"
NEO4J_USER = "neo4j" # Varsayılan
NEO4J_PASSWORD = "test1234"

GEMINI_API_KEY = "AIza...BYsyBKs8"

print("\nBilgiler kaydedildi!")
print(f"Neo4j URI: {NEO4J_URI}")
print(f"Neo4j User: {NEO4J_USER}")
print(f"Gemini Key: AIza...{GEMINI_API_KEY[-8:]}")

Bağlantı bilgilerini tanımlayın:


Bilgiler kaydedildi!
Neo4j URI: neo4j://localhost:7687
Neo4j User: neo4j
Gemini Key: AIza...BYsyBKs8


In [3]:
from neo4j import GraphDatabase


try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    driver.verify_connectivity()
    

    # Veritabanı bilgilerini göster
    with driver.session() as session:
        result = session.run("CALL dbms.components() YIELD name, versions, edition")
        for record in result:
            print(f"Database: {record['name']}")
            print(f"Version: {record['versions'][0]}")
            print(f"Edition: {record['edition']}")

    driver.close()

except Exception as e:
    print(f"BAĞLANTI HATASI: {e}")
    print("\n Kontrol et:")
    print("  1. Neo4j URI doğru mu?")
    print("  2. Şifre doğru mu?")
    print("  3. Neo4j Aura instance çalışıyor mu?")

Database: Neo4j Kernel
Version: 5.26.14
Edition: community


In [4]:
class KGRetriever:
    
    def __init__(self, kg):
        self.kg = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    
    
    def list_all_functions(self):
        with self.kg.session() as session:
            result = session.run("""
                MATCH (m:Module)-[:CONTAINS]->(f:Function)
                RETURN f.name AS FunctionName, 
                       f.docstring AS Description
            """)
            return [record.data() for record in result]
    
    def list_all_concepts(self):
        with self.kg.session() as session:
            result = session.run("""
                MATCH (c:Concept)
                RETURN c.name 
            """)
            return [record.data()["c.name"] for record in result]

    def list_all_libraries(self):
        with self.kg.session() as session:
            result = session.run("""
                MATCH (l:Library)
                RETURN l.name
            """)
            
            return [record.data()["l.name"] for record in result]

    def list_all_modules(self):
        with self.kg.session() as session:
            result = session.run("""
                MATCH (m:Module)
                RETURN m.name
            """)
            return [record.data()["m.name"] for record in result]
    
    def get_all_properties(self):
        functions = [item['FunctionName'] for item in self.list_all_functions()]
        return ("Functions:" + ", ".join(functions) + "\n" +
                "Lİbraries:" + ", ".join(self.list_all_libraries()) + "\n" +
                "Modules:" + ", ".join(self.list_all_modules()) + "\n" +
                "Concepts:" + ", ".join(self.list_all_concepts()))
    
    def custom_query(self, query:str):
        with self.kg.session() as session:
            result = session.run(query)
            return [str(record.data()) for record in result]



retriver = KGRetriever(driver)
graph_data = retriver.list_all_functions()
graph_data

[{'FunctionName': 'save_text',
  'Description': 'Saves list of lines to a text file.'},
 {'FunctionName': 'export_to_json',
  'Description': 'Exports object to a JSON file.'},
 {'FunctionName': 'export_to_csv',
  'Description': 'Exports list of values to a CSV file.'},
 {'FunctionName': 'is_numeric', 'Description': 'Checks if text is numeric.'},
 {'FunctionName': 'to_upper', 'Description': 'Converts text to uppercase.'},
 {'FunctionName': 'debug', 'Description': 'Prints debug message.'},
 {'FunctionName': 'read_first_line',
  'Description': 'Reads the first line of a file.'},
 {'FunctionName': 'load_data',
  'Description': 'Loads data from file if exists.'},
 {'FunctionName': 'min_value', 'Description': 'Returns minimum value.'},
 {'FunctionName': 'max_value', 'Description': 'Returns maximum value.'},
 {'FunctionName': 'compute_statistics',
  'Description': 'Computes count, avg, and stdev.'},
 {'FunctionName': 'filter_short',
  'Description': 'Filters strings shorter than min_len.'},
 

In [6]:
!pip install transformers==4.30.2 sentence-transformers==2.2.2 huggingface-hub==0.16.4

DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 19.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 8.4 MB/s  0:00:00m0:00:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.2/147.2 MB 6.6 MB/s  0:00:22m0:00:0100:01
  Created wheel for sentence-transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=125939 sha256=75748f3d387b7b7c73cf2fd3052f86f724f6d4bd52daace9cddc9dbaf7efbd84
  Stored in directory: /Users/musasahin/Library/Caches/pip/wheels/71/67/06/162a3760c40d74dd40bc855d527008d26341c2b0ecf

  Attempting uninstall: transformers0m━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [huggingface-hub]
    Found existing installation: transformers 4.57.3━━━━━━━━━━ 2/5 [huggingface-hub]
    Uninstalling transformers-4.57.3:╺━━━━━━━━━━━━━━━ 3/5 [transformers]
      Successfully uninstalled transformers-4.57.3m━━━━━━━━━━━━━━━ 3/5 [transformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━ 3/5 [transformers]  DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621
  Attempting uninstall: sentence-transformersm━━━━━━━━━━━━━━━ 3/5 [transformers]
    Found existing installation: sentence-transformers 5.1.2━━ 3/5 [transformers]
    Uninstalling sentence-transformers-5.1.2:0m━━━━━━━━━━━━━━━ 3/5 [transformers]
      Successfully uninstalled sentence-transformers-5.1.2━━━━ 3/5 [transformers]
   ━━━━━━━━━━━━━━━━━━

In [6]:
from sentence_transformers import SentenceTransformer, util

class TextEmbeddingUtil:
    
    def __init__(self, retriever: KGRetriever):
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.functions = retriever.list_all_functions()
        
    def find_similar_functions(self, user_query):
        
        functions = [record["FunctionName"] + " " + record["Description"] for record in self.functions]
        
        function_embeddings = self.model.encode(functions, convert_to_tensor=True)

        query_embedding = self.model.encode(user_query, convert_to_tensor=True)

        cosine_scores = util.cos_sim(query_embedding, function_embeddings)[0]

        similar_functions = []
        for i, score in enumerate(cosine_scores):
            similar_functions.append({
                "function": self.functions[i]['FunctionName'],
                "description": self.functions[i]['Description'],
                "similarity_score": round(float(score), 4)
            })

        similar_functions.sort(key=lambda x: x['similarity_score'], reverse=True)

        return similar_functions
embedding_util = TextEmbeddingUtil(retriver)
embedding_util.find_similar_functions("save to database")

[{'function': 'save_text',
  'description': 'Saves list of lines to a text file.',
  'similarity_score': 0.4737},
 {'function': 'export_to_json',
  'description': 'Exports object to a JSON file.',
  'similarity_score': 0.2606},
 {'function': 'export_to_csv',
  'description': 'Exports list of values to a CSV file.',
  'similarity_score': 0.243},
 {'function': 'clean_data',
  'description': 'Removes empty items.',
  'similarity_score': 0.2141},
 {'function': 'load_data',
  'description': 'Loads data from file if exists.',
  'similarity_score': 0.1932},
 {'function': 'to_upper',
  'description': 'Converts text to uppercase.',
  'similarity_score': 0.0674},
 {'function': 'max_value',
  'description': 'Returns maximum value.',
  'similarity_score': 0.0632},
 {'function': 'read_first_line',
  'description': 'Reads the first line of a file.',
  'similarity_score': 0.0589},
 {'function': 'is_numeric',
  'description': 'Checks if text is numeric.',
  'similarity_score': 0.0529},
 {'function': '

In [15]:
import google.generativeai as genai

class Neo4jRAGSystem:
    def __init__(self, retriever: KGRetriever, embedding_util: TextEmbeddingUtil, gemini_api_key):
        self.retriever = retriever
        self.graph_data = self.retriever.get_all_properties()
        self.embedding_util = embedding_util
        genai.configure(api_key=gemini_api_key)
        self.model = genai.GenerativeModel('gemini-2.5-flash')
        print("RAG Sistemi başlatıldı")

    def pipeline(self, question):
        if "similar" in question and "function" in question:
            wrapped_text = question.replace('', 'function')
            wrapped_text = wrapped_text.replace('', 'similar')
            return self.embedding_util.find_similar_functions(wrapped_text)
        else:
            query = self.get_cypher_query(question)
            print("Cypher query:" + query)
            sub_graph_data = str(self.retriever.custom_query(query))
            print("Sub-graph data:" + sub_graph_data)
            return self.ask(question, sub_graph_data)
    
    def get_cypher_query(self, question):
        
        prompt = f""" Return the cypher query for question. 


Knowlegde Graph Relationship schema:

Module -[CONTAINS]-> Function
Function -[CALLS]-> Function
Module -[IMPORTS]-> Library
Function -[RELATES_TO]-> Concept
Function -[USES_LIBRARY]-> Library

The knowledge graph data:
{self.graph_data}

Question: {question}
ONLY cypher query have to be returned, additional information is forbidden.


"""

        response = self.model.generate_content(
            prompt,
            generation_config=genai.GenerationConfig(
                temperature=0.3,
                max_output_tokens=2048,
            )
        )
        
        query = response.text.replace("\n```", "")
        query = query.replace("```cypher\n","")
        
        return query
    
    
    def ask(self, question, graph_data):
        
        prompt = f"""You are a coding assistant. Answer the questions by using the query-answer pair provided to you. The data consist of question and response of neo4j knowledge graph. 
The knowledge graph data:
{question} -> {graph_data}

Question: {question}
RULES:
1. ONLY use the knowledge graph data provided above.
2. NEVER make up information that is not in the knowledge graph.
3. If there is no information about the question, say, "I do not have information about this."
4. Respond in a natural, sincere, and helpful tone.

"""

        response = self.model.generate_content(
            prompt,
            generation_config=genai.GenerationConfig(
                temperature=0.3,
                max_output_tokens=2048,
            )
        )

        return response.text

# RAG sistemini başlat
rag_system = Neo4jRAGSystem(retriver, embedding_util, GEMINI_API_KEY)

RAG Sistemi başlatıldı


In [8]:
# Test soruları
test_question = "Show similar functions to capital letters?"
    


print(f"\n{'='*70}")
print(test_question)
print("=" * 70)

result = rag_system.pipeline(test_question)

print(f"\n Answer:\n{result}")





Show similar functions to capital letters?

 Answer:
[{'function': 'to_upper', 'description': 'Converts text to uppercase.', 'similarity_score': 0.2411}, {'function': 'max_value', 'description': 'Returns maximum value.', 'similarity_score': 0.2371}, {'function': 'normalize', 'description': 'Normalizes values.', 'similarity_score': 0.2151}, {'function': 'filter_short', 'description': 'Filters strings shorter than min_len.', 'similarity_score': 0.2092}, {'function': 'min_value', 'description': 'Returns minimum value.', 'similarity_score': 0.1995}, {'function': 'is_numeric', 'description': 'Checks if text is numeric.', 'similarity_score': 0.1403}, {'function': 'transform_values', 'description': 'Converts items to their length.', 'similarity_score': 0.1373}, {'function': 'export_to_json', 'description': 'Exports object to a JSON file.', 'similarity_score': 0.1336}, {'function': 'read_first_line', 'description': 'Reads the first line of a file.', 'similarity_score': 0.1165}, {'function': '

In [16]:
test_question = "List all functions that module util contains?"

print(f"\n{'='*70}")
print(test_question)
print("=" * 70)

result = rag_system.pipeline(test_question)

print(f"\n Answer:\n{result}")



List all functions that module util contains?
Cypher query:MATCH (m:Module)-[:CONTAINS]->(f:Function)
WHERE m.name = 'util'
RETURN f.name
Sub-graph data:["{'f.name': 'is_numeric'}", "{'f.name': 'to_upper'}", "{'f.name': 'debug'}"]

 Answer:
The module 'util' contains the following functions: 'is_numeric', 'to_upper', and 'debug'.


In [17]:
test_question = "Which functions call load_data?"

print(f"\n{'='*70}")
print(test_question)
print("=" * 70)

result = rag_system.pipeline(test_question)

print(f"\n Answer:\n{result}")


Which functions call load_data?
Cypher query:MATCH (f1:Function)-[:CALLS]->(f2:Function {name: 'load_data'}) RETURN f1.name
Sub-graph data:["{'f1.name': 'run_pipeline'}", "{'f1.name': 'read_first_line'}"]

 Answer:
Based on the information I have, the functions that call `load_data` are `run_pipeline` and `read_first_line`.


In [18]:
test_question = "Show functions related to ‘file I/O’"

print(f"\n{'='*70}")
print(test_question)
print("=" * 70)

result = rag_system.pipeline(test_question)

print(f"\n Answer:\n{result}")


Show functions related to ‘file I/O’
Cypher query:MATCH (f:Function)-[:RELATES_TO]->(c:Concept) WHERE c.name = 'File I/O' RETURN f.name
Sub-graph data:["{'f.name': 'load_data'}", "{'f.name': 'save_text'}"]

 Answer:
Based on the information I have, the functions related to 'file I/O' are `load_data` and `save_text`.


In [20]:
test_question = "How pipeline and transform_values are related?"

print(f"\n{'='*70}")
print(test_question)
print("=" * 70)

result = rag_system.pipeline(test_question)

print(f"\n Answer:\n{result}")


How pipeline and transform_values are related?
Cypher query:MATCH (f1:Function {name: 'run_pipeline'})-[r]-(f2:Function {name: 'transform_values'}) RETURN f1, r, f2
Sub-graph data:["{'f1': {'file_path': 'main.py', 'docstring': 'Runs the full ETL pipeline.', 'line_number': 6, 'name': 'run_pipeline', 'func_id': 'F16'}, 'r': ({'file_path': 'main.py', 'docstring': 'Runs the full ETL pipeline.', 'line_number': 6, 'name': 'run_pipeline', 'func_id': 'F16'}, 'CALLS', {'file_path': 'processor.py', 'docstring': 'Converts items to their length.', 'line_number': 4, 'name': 'transform_values', 'func_id': 'F13'}), 'f2': {'file_path': 'processor.py', 'docstring': 'Converts items to their length.', 'line_number': 4, 'name': 'transform_values', 'func_id': 'F13'}}"]

 Answer:
Based on the knowledge graph data, the function `run_pipeline` (which is described as running the full ETL pipeline) calls the function `transform_values`.
